# 05. Recommendation System

Content-based + Collaborative + Hybrid.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np


In [ ]:
from src.models.recommendation.train import RecommenderTrainer

trainer = RecommenderTrainer()
content, collab, hybrid = trainer.train_all()

## Find similar properties

In [ ]:
rec = content.recommend_similar(seed_idx=0, top_n=10, filter_label='undervalued')
rec[['borough','price','sqft','walk_score','school_quality_score','similarity']].head()

## Preference-based recommendation

In [ ]:
prefs = {
    'budget_max': 1_200_000,
    'borough': 'BROOKLYN',
    'prefer_safe': True,
    'prefer_school': True,
    'prefer_undervalued': True,
}
rec = content.recommend_from_preferences(prefs, top_n=10)
rec[['borough','price','school_quality_score','crime_rate_per_1k','valuation_label_name','preference_score']]

## Hybrid - content + collaborative

In [ ]:
rec = hybrid.recommend(buyer_id='buyer_0001', seed_property_idx=0, top_n=10)
rec[['borough','price','similarity','collab_score','hybrid_score','valuation_label_name']]

In [ ]:
## Recommender Evaluation

def precision_at_k(recommended_ids, relevant_ids, k):
    """Fraction of top-k recommendations that are relevant."""
    top_k = recommended_ids[:k]
    hits = sum(1 for r in top_k if r in relevant_ids)
    return hits / k

def recall_at_k(recommended_ids, relevant_ids, k):
    """Fraction of relevant items found in top-k."""
    top_k = recommended_ids[:k]
    hits = sum(1 for r in top_k if r in relevant_ids)
    return hits / len(relevant_ids) if relevant_ids else 0

def ndcg_at_k(recommended_ids, relevant_ids, k):
    """Normalized Discounted Cumulative Gain."""
    import math
    top_k = recommended_ids[:k]
    dcg = sum(1/math.log2(i+2) for i, r in enumerate(top_k) if r in relevant_ids)
    ideal = sum(1/math.log2(i+2) for i in range(min(k, len(relevant_ids))))
    return dcg / ideal if ideal > 0 else 0

# Define "relevant" as undervalued properties in Brooklyn under $1.2M
relevant = set(
    content.df[
        (content.df['valuation_label_name'] == 'undervalued') &
        (content.df['borough'] == 'BROOKLYN') &
        (content.df['price'] <= 1_200_000)
    ]['property_id'].tolist()
)

prefs = {'budget_max': 1_200_000, 'borough': 'BROOKLYN',
         'prefer_undervalued': True, 'prefer_school': True}

results_ids = content.recommend_from_preferences(prefs, top_n=20)['property_id'].tolist()

for k in [5, 10, 20]:
    p = precision_at_k(results_ids, relevant, k)
    r = recall_at_k(results_ids, relevant, k)
    n = ndcg_at_k(results_ids, relevant, k)
    print(f"k={k:2d}  Precision@k={p:.3f}  Recall@k={r:.3f}  NDCG@k={n:.3f}")

# Coverage
catalog_size = len(content.df)
recommended_unique = len(set(results_ids))
coverage = recommended_unique / catalog_size
print(f"\nCoverage: {coverage:.4f} ({recommended_unique} unique / {catalog_size} total)")